# 02 — dE/dx Fitting Playground

Interactive exploration of the Landau fitting pipeline.
Try different truncation fractions, bin counts, and fit models
and see the results immediately.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from hibeam import config
from hibeam.io import sim_loader
from hibeam.physics import dedx, fitting
from hibeam.plotting import style, histograms
from hibeam.utils import print_fit_summary

cfg = config.load('../config.yaml')
style.apply(cfg)

## 1. Load data

Change `dataset` below to switch between simulation files.

In [ ]:
# ── Change this to try different datasets ──────────────────────────
dataset = 'krakow'   # Options: 'krakow', 'muon', 'hibeam'
# ───────────────────────────────────────────────────────────────────

path = config.resolve_path(cfg, cfg.paths.simulation[dataset])
data = sim_loader.load_prototpc(str(path))
sim_loader.edep_to_mev(data)

values = dedx.compute_dedx(data, method='sum_edep', min_steps=5)
print(f'Dataset: {dataset}')
print(f'Events after cuts: {len(values):,}')
print(f'Range: {values.min():.4f} – {values.max():.2f} MeV')

## 2. Default fit (from config.yaml)

In [ ]:
result = fitting.fit_landau(
    values,
    truncation=cfg.fitting.truncation,
    n_bins=cfg.fitting.n_bins,
    model=cfg.fitting.model,
)
print_fit_summary(result.as_dict())

histograms.plot_dedx(
    result,
    title=f'{dataset} — default config (trunc={cfg.fitting.truncation}, bins={cfg.fitting.n_bins})',
    cfg=cfg,
)

## 3. Try different truncation fractions

The truncation fraction controls how much of the high-energy Landau
tail is excluded from the fit.  The standard value is 70% (ALICE).
Lower values are more robust to delta-ray contamination;
higher values use more data but are more sensitive to the tail.

In [ ]:
truncations = [0.50, 0.60, 0.70, 0.80, 0.90]

fig, axes = plt.subplots(1, len(truncations), figsize=(5*len(truncations), 5),
                         sharey=True)

mpvs, xis, chi2s = [], [], []

for ax, trunc in zip(axes, truncations):
    r = fitting.fit_landau(values, truncation=trunc, n_bins=80)
    mpvs.append(r.mpv)
    xis.append(r.scale)
    chi2s.append(r.chi2_red)

    bc = r.bin_centers
    ax.bar(bc[r.fit_mask], r.counts[r.fit_mask], width=r.bin_width*0.9,
           color='#2166ac', alpha=0.5)
    ax.bar(bc[~r.fit_mask], r.counts[~r.fit_mask], width=r.bin_width*0.9,
           color='#d73027', alpha=0.3)

    x = np.linspace(bc[0], bc[-1], 500)
    ax.plot(x, fitting.moyal_scaled(x, r.loc, r.scale, r.norm), 'k-', lw=1.5)
    ax.set_title(f'trunc={trunc:.0%}\nMPV={r.mpv:.3f}\n$\\chi^2/\\nu$={r.chi2_red:.2f}',
                 fontsize=9)
    ax.set_xlabel('Edep [MeV]', fontsize=9)

axes[0].set_ylabel('Events / bin')
fig.suptitle(f'{dataset} — truncation scan', fontsize=12)
fig.tight_layout()
plt.show()

# Summary
print(f'{"Truncation":>12}  {"MPV [MeV]":>10}  {"ξ [MeV]":>10}  {"χ²/ndf":>8}')
print('-' * 50)
for t, m, x, c in zip(truncations, mpvs, xis, chi2s):
    print(f'{t:>12.0%}  {m:>10.4f}  {x:>10.4f}  {c:>8.3f}')

## 4. Effect of bin count

In [ ]:
bin_counts = [30, 50, 80, 100, 150, 200]

print(f'{"Bins":>6}  {"MPV":>10}  {"ξ":>10}  {"χ²/ndf":>8}  {"p-value":>8}')
print('-' * 50)
for nb in bin_counts:
    try:
        r = fitting.fit_landau(values, truncation=0.70, n_bins=nb)
        print(f'{nb:>6}  {r.mpv:>10.4f}  {r.scale:>10.4f}  {r.chi2_red:>8.3f}  {r.p_value:>8.3f}')
    except Exception as e:
        print(f'{nb:>6}  FAILED: {e}')

## 5. Bethe-Bloch theory overlay

In [ ]:
from hibeam.physics import bethe_bloch

p = np.logspace(1, 5, 2000)  # 10 MeV/c to 100 GeV/c

fig, ax = plt.subplots(figsize=(10, 6))
for name, props in bethe_bloch.PARTICLES.items():
    bb = bethe_bloch.bethe_bloch(p, props['mass'], props['z'])
    valid = bb > 0
    ax.plot(p[valid], bb[valid], lw=2, color=props['color'], label=name)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Momentum [MeV/c]')
ax.set_ylabel(r'$-\langle dE/dx \rangle$ [MeV/cm]')
ax.set_title('Bethe-Bloch in Ar/CO₂ 90/10')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 6. Compare two datasets side by side

In [ ]:
datasets_to_compare = ['krakow', 'muon']
results = {}

for ds in datasets_to_compare:
    p = config.resolve_path(cfg, cfg.paths.simulation[ds])
    if not p.exists():
        print(f'Skipping {ds}: file not found')
        continue
    d = sim_loader.load_prototpc(str(p))
    sim_loader.edep_to_mev(d)
    v = dedx.compute_dedx(d, method='sum_edep', min_steps=5)
    results[ds] = fitting.fit_landau(v, truncation=0.70, n_bins=80)

if len(results) == 2:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    for ax, (ds, r) in zip([ax1, ax2], results.items()):
        bc = r.bin_centers
        ax.bar(bc, r.counts, width=r.bin_width*0.9, color='#2166ac', alpha=0.5)
        x = np.linspace(bc[0], bc[-1], 500)
        ax.plot(x, fitting.moyal_scaled(x, r.loc, r.scale, r.norm), 'k-', lw=2)
        ax.set_title(f'{ds}\nMPV={r.mpv:.4f} MeV, ξ={r.scale:.4f}', fontsize=11)
        ax.set_xlabel(r'$\sum E_\mathrm{dep}$ [MeV]')
        ax.set_ylabel('Events / bin')
    fig.tight_layout()
    plt.show()